In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", None)

# Loading data

In [2]:
# alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v2.csv")
# weather = pd.read_csv("data/weather/weather_data_preprocessed_v3.csv")
# isw = pd.read_csv("data/isw/isw_data_preprocessed_v5.csv")
# telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

from app.db.database import Database


with Database("app/db/database.db") as db:
    alarms_rows = db.alarms.get()
    weather_rows = db.weather.get()
    isw_rows = db.isw.get()
    telegram_rows = db.telegram.get()

alarms = pd.DataFrame([dict(row) for row in alarms_rows])
weather = pd.DataFrame([dict(row) for row in weather_rows])
isw_raw = pd.DataFrame([dict(row) for row in isw_rows])
telegram = pd.DataFrame([dict(row) for row in telegram_rows])

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Георгій\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Commiting


In [3]:
alarms.head(3)

,region_id,region,time,alarm
0,3,Хмельницька обл.,2022-02-26 16:00:00,1
1,3,Хмельницька обл.,2022-02-26 17:00:00,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1


In [4]:
weather.head(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,0.0,0.0,82.64,-2.6,0.0,0.0,NaN,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,2022-02-24 00:00:00,Kropyvnytskyi
2,2.0,-1.1,80.51,-1.0,0.0,0.0,NaN,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,2022-02-24 00:00:00,Dnipro


In [5]:
isw_raw.head(3)

,id,date,title,url,text
0,1745,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,"February 24, 3:00 pm EST Russian President Vla..."
1,1746,2022-02-24,Ukraine Conflict Update 6,https://understandingwar.org/research/russia-u...,"Institute for the Study of War, Russia Team 9:..."
2,1743,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Kateryna Stepa..."


In [6]:
telegram.head(3)

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2022-02-24 03:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
2,2022-02-24 04:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0


# Preparing data for merging

All data have different granularity, so it need to be unified. I want to make granularity 1 hour, which would be best option for our task.

Key idea is to merge data by time and region id. Each row will have composed key created by `time` and `region_id` columns

## Alarms

In [7]:
alarms.head()

,region_id,region,time,alarm
0,3,Хмельницька обл.,2022-02-26 16:00:00,1
1,3,Хмельницька обл.,2022-02-26 17:00:00,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1
3,3,Хмельницька обл.,2022-02-27 18:00:00,1
4,3,Хмельницька обл.,2022-02-27 19:00:00,1


In [8]:
alarms.info()

<class 'pandas.DataFrame'>
RangeIndex: 185254 entries, 0 to 185253
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   region_id  185254 non-null  int64
 1   region     185254 non-null  str  
 2   time       185254 non-null  str  
 3   alarm      185254 non-null  int64
dtypes: int64(2), str(2)
memory usage: 14.0 MB


Convert `time` column to datetime format

In [9]:
alarms['time'] = pd.to_datetime(alarms['time'])

In [10]:
alarms

,region_id,region,time,alarm
0,3,Хмельницька обл.,2022-02-26 16:00:00,1
1,3,Хмельницька обл.,2022-02-26 17:00:00,1
2,3,Хмельницька обл.,2022-02-27 17:00:00,1
3,3,Хмельницька обл.,2022-02-27 18:00:00,1
4,3,Хмельницька обл.,2022-02-27 19:00:00,1
...,...,...,...,...
185249,31,Київ,2026-04-09 05:00:00,1
185250,31,Київ,2026-04-09 06:00:00,1
185251,31,Київ,2026-04-09 21:00:00,1
185252,31,Київ,2026-04-09 22:00:00,1


In [11]:
# regions = alarms.groupby(['region_id', 'region_city']).size().reset_index().drop(columns=0)
# regions

In [12]:
alarms.isna().sum()

region_id    0
region       0
time         0
alarm        0
dtype: int64

In [13]:
# alarms.start.min(), alarms.end.max()
alarms.time.min(), alarms.time.max()

(Timestamp('2022-02-24 07:00:00'), Timestamp('2026-04-10 16:00:00'))

Make 1 hour granularity

In [14]:
# # create a helper column with the list of hours per alarm
# from app.core.features.alarms_features import explode_by_hour, fix_regions

# alarms = fix_regions(alarms)
# alarms.head()

# alarm_expanded = explode_by_hour(alarms)
# alarm_expanded['time'] = pd.to_datetime(alarm_expanded['time'])
# alarm_expanded.head()

In [15]:
# alarm_expanded.info()

In [16]:
alarms.duplicated(subset=["region_id", "time"]).sum()

np.int64(0)

In [17]:
alarms = alarms.drop_duplicates(subset=["region_id", "time"])

In [18]:
alarms.shape

(185254, 4)

## Weather

In [19]:
weather.head()

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,2022-02-24 00:00:00,Lutsk
1,0.0,0.0,82.64,-2.6,0.0,0.0,NaN,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,2022-02-24 00:00:00,Kropyvnytskyi
2,2.0,-1.1,80.51,-1.0,0.0,0.0,NaN,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,2022-02-24 00:00:00,Dnipro
3,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,2022-02-24 00:00:00,Kyiv
4,4.9,3.1,67.56,-0.6,0.0,0.0,NaN,7.6,29.4,1020.0,24.1,100.0,0.0,Overcast,2022-02-24 00:00:00,Kherson


In [20]:
weather.real_hour_datetime.min(), weather.real_hour_datetime.max()

('2022-02-24 00:00:00', '2026-04-11 23:00:00')

In [21]:
weather.isna().sum()

temp                       0
feelslike                  0
humidity                   0
dew                        0
precip                     0
precipprob                 0
preciptype            717975
windspeed                  0
winddir                    0
pressure                   0
visibility                 0
cloudcover                 0
uvindex                    0
conditions                 0
real_hour_datetime         0
city                       0
dtype: int64

In [22]:
weather.loc[weather.preciptype.isna()].sample(3)

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,real_hour_datetime,city
197548,0.4,-3.0,75.64,-3.4,0.0,0.0,NaN,10.4,167.2,1019.0,20.0,98.4,0.0,Overcast,2023-02-16 22:00:00,Kyiv
343912,8.3,5.5,90.89,6.9,0.0,0.0,NaN,16.9,313.1,1010.0,10.0,98.8,0.0,Overcast,2023-11-09 02:00:00,Cherkasy
803438,11.6,11.6,45.77,0.3,0.0,0.0,NaN,5.8,340.5,1027.0,10.0,75.1,2.0,Partially cloudy,2026-03-06 17:00:00,Lviv


In [23]:
weather["preciptype"] = weather.preciptype.fillna("None")

In [24]:
weather.isna().sum()

temp                  0
feelslike             0
humidity              0
dew                   0
precip                0
precipprob            0
preciptype            0
windspeed             0
winddir               0
pressure              0
visibility            0
cloudcover            0
uvindex               0
conditions            0
real_hour_datetime    0
city                  0
dtype: int64

### Adding `region_id` column

In [25]:
weather_df = weather.copy()

In [26]:
from app.core.features.weather_features import add_region_ids

weather_df = add_region_ids(weather_df)

In [27]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [28]:
weather_df

,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,region_id,time
0,2.4,-1.6,89.18,0.8,0.0,0.0,['snow'],15.5,275.6,1020.0,0.0,91.5,0.0,Overcast,Lutsk,8,2022-02-24 00:00:00
1,0.0,0.0,82.64,-2.6,0.0,0.0,None,2.9,183.0,1021.0,24.1,56.2,0.0,Partially cloudy,Kropyvnytskyi,15,2022-02-24 00:00:00
2,2.0,-1.1,80.51,-1.0,0.0,0.0,None,10.8,4.0,1019.0,10.0,14.1,0.0,Clear,Dnipro,9,2022-02-24 00:00:00
3,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,Kyiv,14,2022-02-24 00:00:00
4,1.9,-0.3,93.08,0.9,0.0,0.0,['snow'],7.4,217.0,1017.9,9.9,99.3,0.0,Overcast,Kyiv,31,2022-02-24 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
846210,5.7,5.7,77.65,2.1,0.0,3.2,None,4.3,229.0,1021.0,24.1,44.0,0.0,Partially cloudy,Mykolaiv,17,2026-04-11 23:00:00
846211,5.9,5.9,78.79,2.5,0.0,0.0,None,2.2,242.9,1018.0,24.1,83.9,0.0,Partially cloudy,Chernivtsi,26,2026-04-11 23:00:00
846212,3.9,3.9,88.67,2.2,0.0,0.0,None,2.5,213.7,1018.0,24.1,99.6,0.0,Overcast,Kyiv,14,2026-04-11 23:00:00
846213,3.9,3.9,88.67,2.2,0.0,0.0,None,2.5,213.7,1018.0,24.1,99.6,0.0,Overcast,Kyiv,31,2026-04-11 23:00:00


## Telegram

In [29]:
telegram.head()

,datetime,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h
0,2022-02-24 02:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2022-02-24 03:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
2,2022-02-24 04:00:00+02:00,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0
3,2022-02-24 05:00:00+02:00,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,2,0
4,2022-02-24 06:00:00+02:00,17,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,18,19,3


In [30]:
telegram.info()

<class 'pandas.DataFrame'>
RangeIndex: 36158 entries, 0 to 36157
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   datetime                   36158 non-null  str  
 1   messages_count             36158 non-null  int64
 2   has_threat_sum             36158 non-null  int64
 3   nlp_артобстрілу            36158 non-null  int64
 4   nlp_бпла                   36158 non-null  int64
 5   nlp_відбій                 36158 non-null  int64
 6   nlp_відбій_тривоги         36158 non-null  int64
 7   nlp_дніпропетровська       36158 non-null  int64
 8   nlp_донецька               36158 non-null  int64
 9   nlp_запорізька             36158 non-null  int64
 10  nlp_нікополь               36158 non-null  int64
 11  nlp_нікополь_нікопольська  36158 non-null  int64
 12  nlp_нікопольська           36158 non-null  int64
 13  nlp_повітряна              36158 non-null  int64
 14  nlp_повітряна_тривога      36158 

In [31]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [32]:
tg_df.time.dtype

datetime64[us, Europe/Kyiv]

In [33]:
tg_df.duplicated(subset="time").sum()

np.int64(0)

## ISW

In [34]:
isw_raw.head()

,id,date,title,url,text
0,1745,2022-02-24,Russia-Ukraine Warning Update: Initial Russian...,https://understandingwar.org/research/russia-u...,"February 24, 3:00 pm EST Russian President Vla..."
1,1746,2022-02-24,Ukraine Conflict Update 6,https://understandingwar.org/research/russia-u...,"Institute for the Study of War, Russia Team 9:..."
2,1743,2022-02-25,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Kateryna Stepa..."
3,1744,2022-02-25,Ukraine Conflict Update 7,https://understandingwar.org/research/russia-u...,Russia Team February 24 ISW published its most...
4,1741,2022-02-26,Russia-Ukraine Warning Update: Russian Offensi...,https://understandingwar.org/research/russia-u...,"Mason Clark, George Barros, and Katya Stepanen..."


In [35]:
isw_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 1758 entries, 0 to 1757
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1758 non-null   int64
 1   date    1758 non-null   str  
 2   title   1758 non-null   str  
 3   url     1758 non-null   str  
 4   text    1758 non-null   str  
dtypes: int64(1), str(4)
memory usage: 49.3 MB


In [36]:
from app.core.features.isw_features import create_features_isw

isw = create_features_isw(isw_raw)

In [37]:
isw.head()

,date,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,centroid_shift_7d,anomaly_count_7d,news_velocity_30d,news_count_30d,anomaly_count_30d,topic_entropy_30d,news_count_7d,dom_cluster_share_7d,centroid_shift_30d,news_velocity_7d,dom_cluster_share_30d,avg_dist_centroid_7d,topic_entropy_7d,avg_dist_centroid_30d
0,2022-02-24,36261,0,0,0,2,0,0,0,0,0,0,NaN,0,0,0,0,NaN,0,NaN,NaN,0,NaN,NaN,NaN,NaN
1,2022-02-25,31129,0,0,0,1,0,0,0,0,0,1,NaN,1,2,2,1,2.162327e-08,2,1.000000,NaN,2,1.000000,0.255692,2.162327e-08,0.255692
2,2022-02-26,37562,0,0,0,1,0,0,0,0,0,1,NaN,1,4,4,1,5.623352e-01,4,0.750000,NaN,4,0.750000,0.300419,5.623352e-01,0.300419
3,2022-02-27,54759,0,0,0,1,0,0,0,0,0,2,NaN,1,6,6,1,6.365142e-01,6,0.666667,NaN,6,0.666667,0.305603,6.365142e-01,0.305603
4,2022-02-28,43324,0,0,0,1,0,0,0,0,0,1,NaN,1,9,9,1,6.869616e-01,9,0.555556,NaN,9,0.555556,0.304577,6.869616e-01,0.304577


In [38]:
isw.date = pd.to_datetime(isw.date)

In [39]:
isw.date

0      2022-02-24
1      2022-02-25
2      2022-02-26
3      2022-02-27
4      2022-02-28
          ...    
1502   2026-04-06
1503   2026-04-07
1504   2026-04-08
1505   2026-04-09
1506   2026-04-10
Name: date, Length: 1507, dtype: datetime64[s]

In [40]:
isw.isna().sum()

date                      0
text_length               0
isw_cluster_0             0
isw_cluster_1             0
isw_cluster_2             0
isw_cluster_3             0
isw_cluster_4             0
isw_cluster_5             0
isw_cluster_6             0
isw_cluster_7             0
isw_cluster_8             0
isw_cluster_9             0
centroid_shift_7d         8
anomaly_count_7d          0
news_velocity_30d         0
news_count_30d            0
anomaly_count_30d         0
topic_entropy_30d         1
news_count_7d             0
dom_cluster_share_7d      1
centroid_shift_30d       31
news_velocity_7d          0
dom_cluster_share_30d     1
avg_dist_centroid_7d      1
topic_entropy_7d          1
avg_dist_centroid_30d     1
dtype: int64

In [41]:
isw.duplicated(subset="date").sum()

np.int64(0)

# Merging

## Creating spine

Idea is to create spine by multiplying hours by regions and then merge everythin to it by time and region_id

In [42]:
timeline = pd.date_range(
    alarms.time.min(),
    alarms.time.max(),
    freq="h"
)

region_ids = alarms.region_id.unique()

expected_length = len(timeline) * len(region_ids)

print(f"Number of hours: {len(timeline)}")
print(f"Number of regions: {len(region_ids)}")
print()
print(f"Expected length of spine: {expected_length}")

Number of hours: 36154
Number of regions: 24

Expected length of spine: 867696


In [43]:
spine = pd.MultiIndex.from_product([region_ids, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [44]:
spine.shape

(867696, 2)

In [45]:
spine.head()

,region_id,time
0,3,2022-02-24 07:00:00
1,3,2022-02-24 08:00:00
2,3,2022-02-24 09:00:00
3,3,2022-02-24 10:00:00
4,3,2022-02-24 11:00:00


In [46]:
spine.info()

<class 'pandas.DataFrame'>
RangeIndex: 867696 entries, 0 to 867695
Data columns (total 2 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   region_id  867696 non-null  int64         
 1   time       867696 non-null  datetime64[us]
dtypes: datetime64[us](1), int64(1)
memory usage: 13.2 MB


## Merging alarms to spine

In [47]:
merged_df = spine.merge(alarms, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [48]:
merged_df

,region_id,time,region,alarm
0,3,2022-02-24 07:00:00,NaN,0
1,3,2022-02-24 08:00:00,NaN,0
2,3,2022-02-24 09:00:00,NaN,0
3,3,2022-02-24 10:00:00,NaN,0
4,3,2022-02-24 11:00:00,NaN,0
...,...,...,...,...
867691,31,2026-04-10 12:00:00,NaN,0
867692,31,2026-04-10 13:00:00,NaN,0
867693,31,2026-04-10 14:00:00,NaN,0
867694,31,2026-04-10 15:00:00,NaN,0


## Merging weather

In [49]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [50]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 867696 entries, 0 to 867695
Data columns (total 19 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   region_id   867696 non-null  int64         
 1   time        867696 non-null  datetime64[us]
 2   region      185254 non-null  str           
 3   alarm       867696 non-null  int64         
 4   temp        845231 non-null  float64       
 5   feelslike   845231 non-null  float64       
 6   humidity    845231 non-null  float64       
 7   dew         845231 non-null  float64       
 8   precip      845231 non-null  float64       
 9   precipprob  845231 non-null  float64       
 10  preciptype  845231 non-null  str           
 11  windspeed   845231 non-null  float64       
 12  winddir     845231 non-null  float64       
 13  pressure    845231 non-null  float64       
 14  visibility  845231 non-null  float64       
 15  cloudcover  845231 non-null  float64       
 16  uvindex     8

## Merging telegram

In [51]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [52]:
tg_df.shape

(36158, 21)

In [53]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [54]:
merged_df.shape

(867696, 39)

## Merging ISW

In [55]:
isw.shape

(1507, 26)

In [56]:
merged_df["date"] = merged_df["time"].dt.date

In [57]:
merged_df.date = pd.to_datetime(merged_df.date)

In [58]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [59]:
merged_df = merged_df.drop(columns="date")

In [60]:
merged_df.shape

(867696, 64)

## Result

In [61]:
merged_df.head()

,region_id,time,region,alarm,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,city,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,centroid_shift_7d,anomaly_count_7d,news_velocity_30d,news_count_30d,anomaly_count_30d,topic_entropy_30d,news_count_7d,dom_cluster_share_7d,centroid_shift_30d,news_velocity_7d,dom_cluster_share_30d,avg_dist_centroid_7d,topic_entropy_7d,avg_dist_centroid_30d
0,3,2022-02-24 07:00:00+02:00,NaN,0,1.2,-1.1,93.72,0.3,0.0,0.0,None,7.2,287.2,1024.0,0.5,100.0,0.0,Overcast,Khmelnytskyi,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,26.0,-3.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1,3,2022-02-24 08:00:00+02:00,NaN,0,1.4,-0.9,95.09,0.7,0.4,100.0,"['rain', 'snow']",7.2,280.0,1024.7,10.0,100.0,1.0,"Snow, Rain, Overcast",Khmelnytskyi,21.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,45.0,47.0,2.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
2,3,2022-02-24 09:00:00+02:00,NaN,0,1.4,-1.4,91.05,0.1,0.0,0.0,None,9.0,295.3,1025.0,23.7,88.5,0.0,Partially cloudy,Khmelnytskyi,20.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,48.0,67.0,0.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
3,3,2022-02-24 10:00:00+02:00,NaN,0,2.0,-0.6,85.35,-0.2,0.0,0.0,None,8.6,311.1,1026.0,23.5,88.7,1.0,Partially cloudy,Khmelnytskyi,19.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0,86.0,-1.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN
4,3,2022-02-24 11:00:00+02:00,NaN,0,2.2,-0.9,96.49,1.7,0.0,0.0,None,10.8,320.0,1025.8,10.0,100.0,1.0,Overcast,Khmelnytskyi,13.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,52.0,99.0,0.0,36261.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN


In [62]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 867696 entries, 0 to 867695
Data columns (total 64 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  867696 non-null  int64                      
 1   time                       867480 non-null  datetime64[us, Europe/Kyiv]
 2   region                     185254 non-null  str                        
 3   alarm                      867696 non-null  int64                      
 4   temp                       845231 non-null  float64                    
 5   feelslike                  845231 non-null  float64                    
 6   humidity                   845231 non-null  float64                    
 7   dew                        845231 non-null  float64                    
 8   precip                     845231 non-null  float64                    
 9   precipprob                 845231 non-null  floa

In [63]:
merged_df.shape

(867696, 64)

In [64]:
print(f"Expected length: {expected_length}")
print(f"Actual length: {merged_df.shape[0]}")

Expected length: 867696
Actual length: 867696


In [65]:
merged_df.region_id.nunique()

24

In [66]:
merged_df.city.nunique()

23

In [67]:
merged_df.region.nunique()

24

# Saving

In [68]:
save_path = Path("data/merged")

merged_df.to_csv(save_path / "merged_v7.csv", index=False)

# Preprocessing merged data

In [69]:
df = merged_df.copy()

In [70]:
df = df.loc[~df.time.isna()]

In [71]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                         0
time                              0
region                       682280
alarm                             0
temp                          22344
feelslike                     22344
humidity                      22344
dew                           22344
precip                        22344
precipprob                    22344
preciptype                    22344
windspeed                     22344
winddir                       22344
pressure                      22344
visibility                    22344
cloudcover                    22344
uvindex                       22344
conditions                    22344
city                          22344
messages_count                    0
has_threat_sum                    0
nlp_артобстрілу                   0
nlp_бпла                          0
nlp_відбій                        0
nlp_відбій_тривоги                0
nlp_дніпропетровська              0
nlp_донецька                      0
nlp_запорізька              

In [72]:
df = df.drop(["city", "region"], axis=1) # region_id column already represents this information

In [73]:
df.loc[df.temp.isna(), "time"].agg(['max', 'min', 'median'])

max      2026-04-08 23:00:00+03:00
min      2022-07-03 21:00:00+03:00
median   2026-03-20 13:00:00+02:00
Name: time, dtype: datetime64[us, Europe/Kyiv]

In [74]:
df["text_length"] = df["text_length"].fillna(0)
df["preciptype"] = df["preciptype"].fillna("None")
        
isw_cluster_cols = [col for col in df.columns if col.startswith("isw_cluster")]
df[isw_cluster_cols] = df[isw_cluster_cols].fillna(0)
    
df["visibility"] = df.groupby('region_id')["visibility"].ffill()
df["uvindex"] = df.groupby("region_id")["uvindex"].ffill()
df['temp'] = df.groupby('region_id')['temp'].ffill(limit=12)
    
df['messages_count'] = df['messages_count'].fillna(0)

# df["year"] = df["time"].dt.year
# df["month"] = df["time"].dt.month
# df["day"] = df["time"].dt.day 
df["hour"] = df["time"].dt.hour
df['day_of_week'] = df['time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df = df.loc[~df.temp.isna()]

In [75]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

region_id                        0
time                             0
alarm                            0
temp                             0
feelslike                      290
humidity                       290
dew                            290
precip                         290
precipprob                     290
preciptype                       0
windspeed                      290
winddir                        290
pressure                       290
visibility                       0
cloudcover                     290
uvindex                          0
conditions                     290
messages_count                   0
has_threat_sum                   0
nlp_артобстрілу                  0
nlp_бпла                         0
nlp_відбій                       0
nlp_відбій_тривоги               0
nlp_дніпропетровська             0
nlp_донецька                     0
nlp_запорізька                   0
nlp_нікополь                     0
nlp_нікополь_нікопольська        0
nlp_нікопольська    

In [76]:
df.loc[df.centroid_shift_30d.isna(), "time"].agg(['min', 'max', 'median'])

min      2022-02-24 07:00:00+02:00
max      2022-03-26 23:00:00+02:00
median   2022-03-11 15:00:00+02:00
Name: time, dtype: datetime64[us, Europe/Kyiv]

missing values only at first month of war, so we can drop them.

In [77]:
df = df.dropna(axis=0)

In [78]:
df.isna().sum().sum()

np.int64(0)

In [79]:
df = df.reset_index(drop=True)

In [80]:
encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)

cat_cols = list(df.select_dtypes(include=["object", "category"]).columns)

df[cat_cols] = encoder.fit_transform(df[cat_cols])

In [81]:
df.head(3)

,region_id,time,alarm,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,centroid_shift_7d,anomaly_count_7d,news_velocity_30d,news_count_30d,anomaly_count_30d,topic_entropy_30d,news_count_7d,dom_cluster_share_7d,centroid_shift_30d,news_velocity_7d,dom_cluster_share_30d,avg_dist_centroid_7d,topic_entropy_7d,avg_dist_centroid_30d,hour,day_of_week,is_weekend
0,3,2022-03-27 00:00:00+02:00,0,7.0,3.1,68.99,1.7,0.0,0.0,0.0,24.5,312.8,1018.0,24.1,55.8,0.0,4.0,19.0,7.0,0.0,0.0,7.0,7.0,2.0,0.0,0.0,0.0,0.0,0.0,6.0,6.0,7.0,7.0,4.0,42.0,352.0,5.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.7,0.533511,2.0,0.62,0.388121,0.610864,0.471807,0,6,1
1,3,2022-03-27 01:00:00+02:00,0,5.7,1.2,44.72,-5.4,0.0,0.0,0.0,26.6,326.8,1019.0,24.1,4.2,0.0,0.0,12.0,8.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,8.0,8.0,8.0,0.0,2.0,37.0,360.0,1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.7,0.533511,2.0,0.62,0.388121,0.610864,0.471807,1,6,1
2,3,2022-03-27 02:00:00+02:00,0,3.8,-0.9,43.48,-7.5,0.0,0.0,0.0,23.8,322.6,1021.0,24.1,0.0,0.0,0.0,23.0,10.0,0.0,0.0,11.0,11.0,2.0,0.0,2.0,0.0,0.0,0.0,10.0,10.0,12.0,11.0,2.0,54.0,378.0,2.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.7,0.533511,2.0,0.62,0.388121,0.610864,0.471807,2,6,1


In [82]:
df.sort_values(by=["region_id", "time"])

,region_id,time,alarm,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,centroid_shift_7d,anomaly_count_7d,news_velocity_30d,news_count_30d,anomaly_count_30d,topic_entropy_30d,news_count_7d,dom_cluster_share_7d,centroid_shift_30d,news_velocity_7d,dom_cluster_share_30d,avg_dist_centroid_7d,topic_entropy_7d,avg_dist_centroid_30d,hour,day_of_week,is_weekend
0,3,2022-03-27 00:00:00+02:00,0,7.0,3.1,68.99,1.7,0.0,0.0,0.0,24.5,312.8,1018.0,24.1,55.8,0.0,4.0,19.0,7.0,0.0,0.0,7.0,7.0,2.0,0.0,0.0,0.0,0.0,0.0,6.0,6.0,7.0,7.0,4.0,42.0,352.0,5.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.700,0.533511,2.0,0.620000,0.388121,0.610864,0.471807,0,6,1
1,3,2022-03-27 01:00:00+02:00,0,5.7,1.2,44.72,-5.4,0.0,0.0,0.0,26.6,326.8,1019.0,24.1,4.2,0.0,0.0,12.0,8.0,0.0,0.0,0.0,0.0,2.0,0.0,2.0,0.0,0.0,0.0,8.0,8.0,8.0,0.0,2.0,37.0,360.0,1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.700,0.533511,2.0,0.620000,0.388121,0.610864,0.471807,1,6,1
2,3,2022-03-27 02:00:00+02:00,0,3.8,-0.9,43.48,-7.5,0.0,0.0,0.0,23.8,322.6,1021.0,24.1,0.0,0.0,0.0,23.0,10.0,0.0,0.0,11.0,11.0,2.0,0.0,2.0,0.0,0.0,0.0,10.0,10.0,12.0,11.0,2.0,54.0,378.0,2.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.700,0.533511,2.0,0.620000,0.388121,0.610864,0.471807,2,6,1
3,3,2022-03-27 04:00:00+03:00,0,2.2,-2.9,47.95,-7.7,0.0,0.0,0.0,23.0,320.1,1021.0,24.1,0.0,0.0,0.0,5.0,1.0,0.0,0.0,5.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0,40.0,382.0,-9.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.700,0.533511,2.0,0.620000,0.388121,0.610864,0.471807,4,6,1
4,3,2022-03-27 05:00:00+03:00,0,1.1,-4.3,52.28,-7.6,0.0,0.0,0.0,22.7,319.1,1022.0,24.1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.0,382.0,-1.0,11717.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.173424,1.0,48.0,50.0,5.0,0.664064,10.0,0.700,0.533511,2.0,0.620000,0.388121,0.610864,0.471807,5,6,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827443,31,2026-04-10 12:00:00+03:00,0,3.0,-0.1,89.87,1.5,0.4,16.1,5.0,11.5,312.6,1015.0,3.8,99.4,2.0,3.0,14.0,4.0,3.0,0.0,3.0,0.0,0.0,0.0,0.0,3.0,3.0,3.0,5.0,5.0,5.0,0.0,0.0,43.0,43.0,-2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.261199,0.0,0.0,34.0,0.0,0.362211,8.0,0.875,0.210439,0.0,0.882353,0.228759,0.376770,0.277995,12,4,0
827444,31,2026-04-10 13:00:00+03:00,0,3.7,0.7,86.15,1.6,0.3,16.1,5.0,11.9,311.8,1015.0,8.7,96.3,3.0,3.0,23.0,4.0,3.0,0.0,10.0,8.0,0.0,0.0,4.0,9.0,9.0,9.0,3.0,3.0,5.0,8.0,0.0,66.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.261199,0.0,0.0,34.0,0.0,0.362211,8.0,0.875,0.210439,0.0,0.882353,0.228759,0.376770,0.277995,13,4,0
827445,31,2026-04-10 14:00:00+03:00,0,3.9,1.0,84.95,1.6,0.3,16.1,5.0,11.2,309.6,1015.0,7.0,96.5,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.261199,0.0,0.0,34.0,0.0,0.362211,8.0,0.875,0.210439,0.0,0.882353,0.228759,0.376770,0.277995,14,4,0
827446,31,2026-04-10 15:00:00+03:00,0,4.3,1.5,83.79,1.8,0.3,16.1,5.0,11.2,315.0,1015.0,7.7,99.1,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

In [83]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 827448 entries, 0 to 827447
Data columns (total 65 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  827448 non-null  int64                      
 1   time                       827448 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      827448 non-null  int64                      
 3   temp                       827448 non-null  float64                    
 4   feelslike                  827448 non-null  float64                    
 5   humidity                   827448 non-null  float64                    
 6   dew                        827448 non-null  float64                    
 7   precip                     827448 non-null  float64                    
 8   precipprob                 827448 non-null  float64                    
 9   preciptype                 827448 non-null  floa

In [84]:
cols_to_int = ['messages_count', 'has_threat_sum',
       'nlp_артобстрілу', 'nlp_бпла', 'nlp_відбій', 'nlp_відбій_тривоги',
       'nlp_дніпропетровська', 'nlp_донецька', 'nlp_запорізька',
       'nlp_нікополь', 'nlp_нікополь_нікопольська', 'nlp_нікопольська',
       'nlp_повітряна', 'nlp_повітряна_тривога', 'nlp_тривога', 'nlp_тривоги',
       'nlp_харківська', 'msg_count_last_3h', 'msg_count_last_24h',
       'threat_diff_1h', 'day_of_week', 'is_weekend',
       'text_length', 'isw_cluster_0', 'isw_cluster_1', 'isw_cluster_2',
       'isw_cluster_3', 'isw_cluster_4', 'isw_cluster_5', 'isw_cluster_6',
       'isw_cluster_7', 'isw_cluster_8', 'isw_cluster_9', "news_count_7d", 
        "news_count_30d", "anomaly_count_7d", "anomaly_count_30d",
        "news_velocity_7d", "news_velocity_30d"]

df[cols_to_int] = df[cols_to_int].astype(int)

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 827448 entries, 0 to 827447
Data columns (total 65 columns):
 #   Column                     Non-Null Count   Dtype                      
---  ------                     --------------   -----                      
 0   region_id                  827448 non-null  int64                      
 1   time                       827448 non-null  datetime64[us, Europe/Kyiv]
 2   alarm                      827448 non-null  int64                      
 3   temp                       827448 non-null  float64                    
 4   feelslike                  827448 non-null  float64                    
 5   humidity                   827448 non-null  float64                    
 6   dew                        827448 non-null  float64                    
 7   precip                     827448 non-null  float64                    
 8   precipprob                 827448 non-null  float64                    
 9   preciptype                 827448 non-null  floa

In [86]:
df.region_id.unique()

array([ 3,  4,  5,  8,  9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 20, 21, 22,
       23, 24, 25, 26, 27, 28, 31])

# Extracting features

Calculate number of alarms in all regions by hour

In [87]:
all_alarms = df.groupby("time")["alarm"].sum().rename("num_alarms").reset_index()

df = pd.merge(df, all_alarms, on="time", how="left")
# df["other_alarms_count"] = df["num_alarms"] - df["alarm"]
for i in range(12):
    df[f"alarms_count_{i+1}h_ago"] = df["num_alarms"].shift(i+1)

df = df.drop(columns="num_alarms")

Add lag for 1-24 hours

In [88]:
for i in range(24):
    df[f"alarm_status_{i+1}h_ago"] = df.groupby("region_id")["alarm"].shift(i+1)

In [89]:
df = df.dropna(axis=0).reset_index(drop=True)

In [90]:
df.head()

,region_id,time,alarm,temp,feelslike,humidity,dew,precip,precipprob,preciptype,windspeed,winddir,pressure,visibility,cloudcover,uvindex,conditions,messages_count,has_threat_sum,nlp_артобстрілу,nlp_бпла,nlp_відбій,nlp_відбій_тривоги,nlp_дніпропетровська,nlp_донецька,nlp_запорізька,nlp_нікополь,nlp_нікополь_нікопольська,nlp_нікопольська,nlp_повітряна,nlp_повітряна_тривога,nlp_тривога,nlp_тривоги,nlp_харківська,msg_count_last_3h,msg_count_last_24h,threat_diff_1h,text_length,isw_cluster_0,isw_cluster_1,isw_cluster_2,isw_cluster_3,isw_cluster_4,isw_cluster_5,isw_cluster_6,isw_cluster_7,isw_cluster_8,isw_cluster_9,centroid_shift_7d,anomaly_count_7d,news_velocity_30d,news_count_30d,anomaly_count_30d,topic_entropy_30d,news_count_7d,dom_cluster_share_7d,centroid_shift_30d,news_velocity_7d,dom_cluster_share_30d,avg_dist_centroid_7d,topic_entropy_7d,avg_dist_centroid_30d,hour,day_of_week,is_weekend,alarms_count_1h_ago,alarms_count_2h_ago,alarms_count_3h_ago,alarms_count_4h_ago,alarms_count_5h_ago,alarms_count_6h_ago,alarms_count_7h_ago,alarms_count_8h_ago,alarms_count_9h_ago,alarms_count_10h_ago,alarms_count_11h_ago,alarms_count_12h_ago,alarm_status_1h_ago,alarm_status_2h_ago,alarm_status_3h_ago,alarm_status_4h_ago,alarm_status_5h_ago,alarm_status_6h_ago,alarm_status_7h_ago,alarm_status_8h_ago,alarm_status_9h_ago,alarm_status_10h_ago,alarm_status_11h_ago,alarm_status_12h_ago,alarm_status_13h_ago,alarm_status_14h_ago,alarm_status_15h_ago,alarm_status_16h_ago,alarm_status_17h_ago,alarm_status_18h_ago,alarm_status_19h_ago,alarm_status_20h_ago,alarm_status_21h_ago,alarm_status_22h_ago,alarm_status_23h_ago,alarm_status_24h_ago
0,3,2022-03-28 01:00:00+03:00,0,-0.1,-1.8,48.02,-9.8,0.0,0.0,0.0,5.0,291.3,1030.0,24.1,1.6,0.0,0.0,13,5,0,0,4,4,0,0,0,0,0,0,5,5,5,4,0,86,289,-3,13953,0,0,0,0,0,0,0,0,0,1,0.179787,1,45,49,5,0.657529,10,0.7,0.484262,2,0.632653,0.386532,0.610864,0.465249,1,0,0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3,2022-03-28 02:00:00+03:00,0,-0.6,-0.6,51.39,-9.4,0.0,0.0,0.0,4.7,271.0,1030.0,24.1,2.5,0.0,0.0,8,2,0,0,3,3,0,0,0,0,0,0,1,1,1,4,0,45,285,-3,13953,0,0,0,0,0,0,0,0,0,1,0.179787,1,45,49,5,0.657529,10,0.7,0.484262,2,0.632653,0.386532,0.610864,0.465249,2,0,0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2022-03-28 03:00:00+03:00,0,-0.8,-4.5,59.99,-7.6,0.0,0.0,0.0,10.8,270.0,1029.4,10.0,40.0,0.0,4.0,7,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,28,269,-2,13953,0,0,0,0,0,0,0,0,0,1,0.179787,1,45,49,5,0.657529,10,0.7,0.484262,2,0.632653,0.386532,0.610864,0.465249,3,0,0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2022-03-28 04:00:00+03:00,0,-1.3,-1.3,56.25,-8.9,0.0,0.0,0.0,4.3,228.8,1029.0,24.1,18.8,0.0,0.0,11,10,0,0,0,0,2,0,2,0,0,0,9,9,9,0,2,26,275,10,13953,0,0,0,0,0,0,0,0,0,1,0.179787,1,45,49,5,0.657529,10,0.7,0.484262,2,0.632653,0.386532,0.610864,0.465249,4,0,0,8.0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3,2022-03-28 05:00:00+03:00,0,-1.3,-1.3,56.69,-8.8,0.0,0.0,0.0,3.6,199.3,1029.0,24.1,87.8,0.0,4.0,9,1,0,0,6,6,2,0,2,0,0,0,1,1,2,6,2,27,283,-9,13953,0,0,0,0,0,0,0,0,0,1,0.179787,1,45,49,5,0.657529,10,0.7,0.484262,2,0.632653,0.386532,0.610864,0.465249,5,0,0,9.0,8.0,2.0,4.0,5.0,10.0,24.0,15.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Saving result

In [91]:
df.to_csv(save_path / "merged_preprocessed.csv", index=False)

In [92]:
import joblib


model_path = Path("app/models/preprocessing")
if not model_path.exists():
    model_path.mkdir(parents=True, exist_ok=True)

joblib.dump(encoder, model_path / "merged_df_encoder.joblib")

['app\\models\\preprocessing\\merged_df_encoder.joblib']

In [93]:
cat_cols

['preciptype', 'conditions']